# Recertification KYC — ZIP multi-clients, extraction schema-free, avec PaddleOCR-VL local

## Mises à jour par rapport à la version précédente

1. **Noms de fichiers réels** : chaque dossier client contient `JUSTIFICATIF IDENTITE.PDF` et `JUSTIFICATIF DOMICILE.PDF` (majuscules, avec espace) — le repérage reste tolérant à la casse/accents/séparateurs pour absorber d'éventuelles variantes.
2. **PaddleOCR-VL local** (`ModelHub-model-huggingface-PaddlePaddle/PaddleOCR-VL`) remplace l'ancienne approche « PaddleOCR pip + essai de packs linguistiques ». PaddleOCR-VL est un modèle **vision-langage dédié à l'OCR/parsing de documents**, nativement multi-langue (~100 langues) et robuste aux mises en page complexes — il n'a donc plus besoin qu'on lui fasse deviner la langue.

## Rôle de PaddleOCR-VL dans le pipeline

Il n'est **pas** utilisé pour remplacer l'extraction de champs par Qwen2.5-VL/InternVL (schema-free), mais comme **troisième source de vérification indépendante** :
- Il produit une **transcription brute fidèle** du document (texte + mise en page).
- On vérifie ensuite que chaque valeur de champ extraite par Qwen/InternVL **apparaît bien** dans cette transcription brute → détecte les hallucinations éventuelles des VLM d'extraction, un point important en contexte KYC.

## Architecture globale

1. Décompression ZIP + parcours des dossiers clients
2. Repérage de `JUSTIFICATIF IDENTITE.PDF` et `JUSTIFICATIF DOMICILE.PDF`
3. Conversion PDF → images (PyMuPDF, multi-pages)
4. Prétraitement image (rotation grossière + skew fin + débruitage/contraste)
5. **OCR brut multi-langue via PaddleOCR-VL local** (remplace l'ancien multi-pack)
6. Extraction schema-free par Qwen2.5-VL et InternVL (aucun champ prédéfini)
7. Réconciliation floue Qwen ↔ InternVL
8. **Vérification croisée des valeurs contre la transcription PaddleOCR-VL**
9. Cross-vérification d'identité entre les deux documents d'un même client
10. Traitement par lot sur tout le ZIP → export JSON/CSV


In [ ]:
import os
import re
import io
import json
import zipfile
import shutil
import unicodedata
import difflib
from pathlib import Path
from typing import Optional

import cv2
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
from PIL import Image
import torch


## 1. Configuration : chemins ModelHub (Qwen2.5-VL, InternVL, PaddleOCR-VL) et archive ZIP

In [ ]:
MODEL_PATHS = {
    "qwen_vlm": "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-instruct/main",
    "internvl": "/domino/edv/modelhub/ModelHub-model-huggingface-OpenGVLab/InternVL2_5-8B/main",
    "paddleocr_vl": "/domino/edv/modelhub/ModelHub-model-huggingface-PaddlePaddle/PaddleOCR-VL/main",
}

ZIP_PATH = "/mnt/user-data/uploads/dossiers_clients.zip"   # a adapter
EXTRACT_DIR = "/home/work/kyc_extraction"                  # dossier de travail (scratch)
OUTPUT_DIR = "/mnt/user-data/outputs"

# Noms de fichiers cibles reels + variantes tolerees (comparaison normalisee :
# minuscule, sans accents, espaces/underscores ignores)
TARGET_FILES = {
    "id_card": ["justificatifidentite", "carteidentite", "cni", "idcard", "national id"],
    "residence": ["justificatifdomicile", "residence", "proofofaddress", "residencepermit"],
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilise : {DEVICE}")
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Décompression du ZIP et découverte des dossiers clients

In [ ]:
def extract_zip(zip_path: str, dest_dir: str) -> str:
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest_dir)
    return dest_dir


def list_client_folders(root_dir: str) -> dict:
    """Retourne {client_id: chemin_dossier}. Gere le cas ou le zip contient
    un dossier racine unique englobant tous les dossiers clients."""
    entries = [e for e in Path(root_dir).iterdir() if e.is_dir()]
    if len(entries) == 1 and any(sub.is_dir() for sub in entries[0].iterdir()):
        entries = [e for e in entries[0].iterdir() if e.is_dir()]
    return {e.name: str(e) for e in entries}


root = extract_zip(ZIP_PATH, EXTRACT_DIR)
client_folders = list_client_folders(root)
print(f"{len(client_folders)} dossiers clients detectes.")
for cid, path in list(client_folders.items())[:5]:
    print(f"  - client_id={cid}  ->  {path}")


## 3. Repérage de `JUSTIFICATIF IDENTITE.PDF` et `JUSTIFICATIF DOMICILE.PDF`

Comparaison normalisée (casse, accents, espaces/underscores ignorés) : `JUSTIFICATIF IDENTITE.PDF`
et `justificatif_identite.pdf` sont ainsi traités de façon identique.


In [ ]:
def normalize_filename(name: str) -> str:
    name = os.path.splitext(name)[0]
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    name = re.sub(r"[\s_\-]+", "", name.lower())
    return name


def find_target_pdfs(client_folder: str) -> dict:
    """Retourne {"id_card": chemin_ou_None, "residence": chemin_ou_None}."""
    found = {"id_card": None, "residence": None}
    for f in Path(client_folder).glob("*.[pP][dD][fF]"):
        norm = normalize_filename(f.name)
        for doc_key, aliases in TARGET_FILES.items():
            if found[doc_key] is not None:
                continue
            if any(normalize_filename(alias) in norm for alias in aliases):
                found[doc_key] = str(f)
    return found


missing_report = []
client_pdfs = {}
for cid, folder in client_folders.items():
    pdfs = find_target_pdfs(folder)
    client_pdfs[cid] = pdfs
    if pdfs["id_card"] is None or pdfs["residence"] is None:
        missing_report.append({"client_id": cid, **{k: (v is not None) for k, v in pdfs.items()}})

print(f"Clients avec un ou deux documents manquants : {len(missing_report)}")
pd.DataFrame(missing_report).head(10)


## 4. Conversion PDF → images (PyMuPDF), gère les PDF multi-pages

In [ ]:
def pdf_to_images(pdf_path: str, dpi: int = 300) -> list:
    images = []
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    doc = fitz.open(pdf_path)
    for page in doc:
        pix = page.get_pixmap(matrix=matrix)
        img_bytes = pix.tobytes("png")
        img_arr = np.frombuffer(img_bytes, dtype=np.uint8)
        img_bgr = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
        images.append(img_bgr)
    doc.close()
    return images


## 5. Prétraitement image (rotation grossière, skew fin, débruitage/contraste)

In [ ]:
import math

def denoise_and_enhance(img_bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=10, templateWindowSize=7, searchWindowSize=21)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)


def detect_coarse_rotation_osd(img_bgr: np.ndarray) -> int:
    try:
        import pytesseract
        osd = pytesseract.image_to_osd(img_bgr)
        angle = int([l for l in osd.split("\n") if "Rotate" in l][0].split(":")[1].strip())
        return angle
    except Exception as e:
        print(f"[WARN] OSD indisponible ({e}), rotation grossiere non detectee.")
        return 0


def rotate_image(img_bgr: np.ndarray, angle: float) -> np.ndarray:
    if angle == 0:
        return img_bgr
    (h, w) = img_bgr.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos = np.abs(M[0, 0]); sin = np.abs(M[0, 1])
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))
    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]
    return cv2.warpAffine(img_bgr, M, (new_w, new_h), borderValue=(255, 255, 255))


def detect_and_correct_skew(img_bgr: np.ndarray, max_angle: float = 15.0) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=100,
                             minLineLength=gray.shape[1] // 4, maxLineGap=20)
    if lines is None:
        return img_bgr
    angles = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = math.degrees(math.atan2(y2 - y1, x2 - x1))
        if -max_angle <= angle <= max_angle:
            angles.append(angle)
    if not angles:
        return img_bgr
    median_angle = float(np.median(angles))
    return img_bgr if abs(median_angle) < 0.5 else rotate_image(img_bgr, median_angle)


def preprocess_image(img_bgr: np.ndarray) -> np.ndarray:
    coarse_angle = detect_coarse_rotation_osd(img_bgr)
    img_bgr = rotate_image(img_bgr, coarse_angle)
    img_bgr = detect_and_correct_skew(img_bgr)
    img_bgr = denoise_and_enhance(img_bgr)
    return img_bgr


## 6. Chargement des trois modèles depuis le ModelHub local

- **Qwen2.5-VL** et **InternVL** : extraction schema-free (double moteur, comme précédemment)
- **PaddleOCR-VL** : transcription brute de référence, utilisée en vérification (pas d'extraction de champs)

⚠️ PaddleOCR-VL expose une API `AutoModel`/`AutoProcessor` avec `trust_remote_code=True`,
similaire à Qwen2.5-VL. Si la version installée localement diffère légèrement (nom de
méthode `generate` vs `chat`), ajuster `extract_raw_text_with_paddleocr_vl` en conséquence
en consultant le `README`/`modeling_*.py` livré avec le modèle sur le ModelHub.


In [ ]:
from transformers import AutoModelForCausalLM, AutoModel, AutoProcessor, AutoTokenizer
from transformers import Qwen2_5_VLForConditionalGeneration

# --- Qwen2.5-VL : extraction schema-free (source primaire) ---
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATHS["qwen_vlm"],
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    local_files_only=True,
)
qwen_processor = AutoProcessor.from_pretrained(MODEL_PATHS["qwen_vlm"], local_files_only=True)

# --- InternVL : second avis / cross-check ---
internvl_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATHS["internvl"],
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    local_files_only=True,
    trust_remote_code=True,
)
internvl_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATHS["internvl"], local_files_only=True, trust_remote_code=True
)

# --- PaddleOCR-VL : transcription brute de verification (multi-langue nativement) ---
paddleocr_vl_processor = AutoProcessor.from_pretrained(
    MODEL_PATHS["paddleocr_vl"], local_files_only=True, trust_remote_code=True
)
paddleocr_vl_model = AutoModel.from_pretrained(
    MODEL_PATHS["paddleocr_vl"],
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    local_files_only=True,
    trust_remote_code=True,
)

print("Les trois modeles sont charges (Qwen2.5-VL, InternVL, PaddleOCR-VL).")


## 7. Transcription brute via PaddleOCR-VL (vérification, pas d'extraction de champs)

On demande une transcription complète et fidèle du document (texte, sans reformulation ni
interprétation), utilisée ensuite uniquement pour **vérifier la présence** des valeurs
extraites par Qwen/InternVL.


In [ ]:
PADDLEOCR_VL_PROMPT = "OCR:"  # declencheur standard de transcription pour PaddleOCR-VL
# Si le modele livre attend un autre declencheur (ex. balise speciale de type
# "<|ocr_start|>"), l'ajuster ici selon la documentation fournie avec le modele.

def extract_raw_text_with_paddleocr_vl(img_pil: Image.Image) -> str:
    try:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": img_pil},
                {"type": "text", "text": PADDLEOCR_VL_PROMPT},
            ],
        }]
        text_prompt = paddleocr_vl_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = paddleocr_vl_processor(
            text=[text_prompt], images=[img_pil], return_tensors="pt"
        ).to(paddleocr_vl_model.device)
        with torch.no_grad():
            output_ids = paddleocr_vl_model.generate(**inputs, max_new_tokens=2048, do_sample=False)
        output_text = paddleocr_vl_processor.batch_decode(
            output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
        )[0]
        return output_text.strip()
    except Exception as e:
        print(f"[WARN] Echec transcription PaddleOCR-VL : {e}")
        return ""


def estimate_text_quality(raw_text: str) -> float:
    """Heuristique de qualite de transcription (proxy en l'absence de score de confiance
    natif) : proportion de caracteres alphanumeriques + longueur normalisee."""
    if not raw_text:
        return 0.0
    alnum_ratio = sum(c.isalnum() for c in raw_text) / max(len(raw_text), 1)
    length_score = min(len(raw_text) / 500.0, 1.0)
    return round(0.6 * alnum_ratio + 0.4 * length_score, 3)


## 8. Extraction **schema-free** par Qwen2.5-VL et InternVL (inchangé sur le principe)

Aucun champ prédéfini : le modèle liste lui-même les champs observés sur le document.


In [ ]:
OPEN_EXTRACTION_PROMPT = """Tu es un expert en analyse de documents d'identite et justificatifs
administratifs, de tous pays et dans toutes les langues.

Observe attentivement ce document (il peut etre mal scanne, etre une photocopie, legerement
de travers, ou dans une langue que tu ne maitrises pas parfaitement).

Ne suppose AUCUN format ni AUCUNE liste de champs predefinie : identifie uniquement les
champs reellement visibles sur CE document, tels qu'ils apparaissent.

Reponds UNIQUEMENT avec un objet JSON valide, structure suivante (structure fixe,
mais le CONTENU des champs est entierement libre) :

{
  "type_document_detecte": "ex: carte d'identite nationale, passeport, titre de sejour, facture, attestation de domicile, ...",
  "pays_emetteur_estime": "",
  "langue_document": "",
  "champs": [
    {"libelle_original": "texte tel qu'il apparait sur le document", "libelle_normalise_fr": "traduction/normalisation en francais, snake_case", "valeur": "valeur lue"},
    ...
  ],
  "remarques": "toute anomalie constatee : illisible, incoherent, incomplet, etc."
}

Liste TOUS les champs que tu peux lire, sans te limiter a un sous-ensemble. Ne donne
aucun texte en dehors du JSON."""


def safe_json_parse(text: str) -> dict:
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        return {"_parse_error": True, "_raw_output": text}
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return {"_parse_error": True, "_raw_output": text}


def extract_with_qwen(img_pil: Image.Image) -> dict:
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img_pil},
            {"type": "text", "text": OPEN_EXTRACTION_PROMPT},
        ],
    }]
    text_prompt = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_processor(text=[text_prompt], images=[img_pil], return_tensors="pt").to(qwen_model.device)
    with torch.no_grad():
        output_ids = qwen_model.generate(**inputs, max_new_tokens=768, do_sample=False)
    output_text = qwen_processor.batch_decode(
        output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]
    return safe_json_parse(output_text)


def extract_with_internvl(img_pil: Image.Image) -> dict:
    pixel_values = internvl_model.load_image(img_pil) if hasattr(internvl_model, "load_image") else None
    if pixel_values is not None:
        pixel_values = pixel_values.to(internvl_model.device, dtype=internvl_model.dtype)
        response = internvl_model.chat(internvl_tokenizer, pixel_values, OPEN_EXTRACTION_PROMPT, dict(max_new_tokens=768))
    else:
        response = "{}"
    return safe_json_parse(response)


## 9. Réconciliation floue Qwen ↔ InternVL + vérification contre PaddleOCR-VL

Deux niveaux de vérification :
1. **Qwen ↔ InternVL** : appariement des champs par similarité de libellé (`difflib`), comparaison des valeurs.
2. **Valeur ↔ transcription PaddleOCR-VL** : chaque valeur retenue doit être **retrouvable** (après normalisation) dans la transcription brute de PaddleOCR-VL — sinon elle est marquée comme non vérifiée (risque d'hallucination du VLM d'extraction).


In [ ]:
def normalize_text(v) -> str:
    if v is None:
        return ""
    v = unicodedata.normalize("NFKD", str(v)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", v.strip().lower())


def normalize_no_space(v) -> str:
    return re.sub(r"\s+", "", normalize_text(v))


def match_fields(champs_a: list, champs_b: list, threshold: float = 0.72) -> list:
    used_b = set()
    matches = []
    for fa in champs_a:
        label_a = normalize_text(fa.get("libelle_normalise_fr"))
        best_j, best_ratio = None, 0.0
        for j, fb in enumerate(champs_b):
            if j in used_b:
                continue
            label_b = normalize_text(fb.get("libelle_normalise_fr"))
            ratio = difflib.SequenceMatcher(None, label_a, label_b).ratio()
            if ratio > best_ratio:
                best_ratio, best_j = ratio, j
        if best_j is not None and best_ratio >= threshold:
            used_b.add(best_j)
            matches.append((fa, champs_b[best_j], best_ratio))
    return matches


def verify_values_against_ocr(champs: list, ocr_raw_text: str) -> dict:
    """Verifie que chaque valeur de champ est retrouvable dans la transcription PaddleOCR-VL."""
    ocr_norm = normalize_no_space(ocr_raw_text)
    verified, unverified = [], []
    for c in champs:
        val = normalize_no_space(c.get("valeur"))
        if not val:
            continue
        if len(val) >= 3 and val in ocr_norm:
            verified.append(c)
        else:
            unverified.append(c)
    total = len(verified) + len(unverified)
    rate = (len(verified) / total) if total > 0 else None
    return {"taux_verification_ocr": round(rate, 2) if rate is not None else None,
            "champs_non_retrouves_dans_ocr": unverified}


def reconcile_extractions(qwen_out: dict, internvl_out: dict, ocr_raw_text: str) -> dict:
    champs_a = qwen_out.get("champs", []) if isinstance(qwen_out.get("champs"), list) else []
    champs_b = internvl_out.get("champs", []) if isinstance(internvl_out.get("champs"), list) else []

    matches = match_fields(champs_a, champs_b)
    disagreements, agreements = [], []
    for fa, fb, sim in matches:
        va, vb = normalize_text(fa.get("valeur")), normalize_text(fb.get("valeur"))
        entry = {
            "libelle": fa.get("libelle_normalise_fr"),
            "valeur_qwen": fa.get("valeur"),
            "valeur_internvl": fb.get("valeur"),
            "similarite_libelle": round(sim, 2),
        }
        (agreements if va == vb else disagreements).append(entry)

    n_matched = len(matches)
    n_total = max(len(champs_a), len(champs_b), 1)
    coverage = n_matched / n_total
    agreement_rate = (len(agreements) / n_matched) if n_matched > 0 else 0.0

    ocr_verification = verify_values_against_ocr(champs_a, ocr_raw_text)
    ocr_rate = ocr_verification["taux_verification_ocr"] if ocr_verification["taux_verification_ocr"] is not None else 0.5

    confidence = round(coverage * agreement_rate * (0.5 + 0.5 * ocr_rate), 3)
    statut = "certifie" if (confidence >= 0.5 and len(disagreements) == 0 and ocr_rate >= 0.7) else "a_revoir_manuellement"

    return {
        "champs_qwen": champs_a,
        "champs_internvl": champs_b,
        "champs_en_accord": agreements,
        "champs_en_desaccord": disagreements,
        "couverture_appariement": round(coverage, 2),
        "taux_accord": round(agreement_rate, 2),
        "verification_paddleocr_vl": ocr_verification,
        "confiance": confidence,
        "statut": statut,
    }


## 10. Cross-vérification d'identité entre `JUSTIFICATIF IDENTITE.PDF` et `JUSTIFICATIF DOMICILE.PDF`

In [ ]:
NAME_LABEL_HINTS = ["nom", "prenom", "nom complet", "name", "surname", "full name",
                    "nombre", "apellido", "nachname", "vorname", "cognome", "nome"]

def extract_name_like_fields(champs: list) -> list:
    found = []
    for c in champs:
        label = normalize_text(c.get("libelle_normalise_fr")) + " " + normalize_text(c.get("libelle_original"))
        if any(hint in label for hint in NAME_LABEL_HINTS):
            found.append(c.get("valeur"))
    return found


def cross_check_identity(id_card_champs: list, residence_champs: list) -> dict:
    names_id = [normalize_text(n) for n in extract_name_like_fields(id_card_champs) if n]
    names_res = [normalize_text(n) for n in extract_name_like_fields(residence_champs) if n]

    if not names_id or not names_res:
        return {"verifiable": False, "coherent": None,
                "detail": "champs nominatifs introuvables sur l'un des deux documents"}

    best_ratio = 0.0
    for n1 in names_id:
        for n2 in names_res:
            best_ratio = max(best_ratio, difflib.SequenceMatcher(None, n1, n2).ratio())

    return {
        "verifiable": True,
        "coherent": best_ratio >= 0.75,
        "similarite_max": round(best_ratio, 2),
        "noms_carte_identite": names_id,
        "noms_residence": names_res,
    }


## 11. Pipeline complet pour un document (PDF → images → prétraitement → triple vérification)

In [ ]:
def process_pdf_document(pdf_path: str) -> dict:
    """Traite un PDF potentiellement multi-pages : extraction + verification par page,
    puis fusion des champs (utile pour recto/verso)."""
    pages = pdf_to_images(pdf_path)
    all_qwen_champs, all_internvl_champs = [], []
    all_ocr_text = []
    ocr_quality_scores = []
    doc_type_guess, pays_guess, langue_guess = None, None, None

    for page_bgr in pages:
        corrected = preprocess_image(page_bgr)
        img_pil = Image.fromarray(cv2.cvtColor(corrected, cv2.COLOR_BGR2RGB))

        ocr_text = extract_raw_text_with_paddleocr_vl(img_pil)
        all_ocr_text.append(ocr_text)
        ocr_quality_scores.append(estimate_text_quality(ocr_text))

        qwen_out = extract_with_qwen(img_pil)
        internvl_out = extract_with_internvl(img_pil)

        all_qwen_champs.extend(qwen_out.get("champs", []) if isinstance(qwen_out.get("champs"), list) else [])
        all_internvl_champs.extend(internvl_out.get("champs", []) if isinstance(internvl_out.get("champs"), list) else [])

        doc_type_guess = doc_type_guess or qwen_out.get("type_document_detecte")
        pays_guess = pays_guess or qwen_out.get("pays_emetteur_estime")
        langue_guess = langue_guess or qwen_out.get("langue_document")

    full_ocr_text = "\n".join(all_ocr_text)
    reconciliation = reconcile_extractions(
        {"champs": all_qwen_champs}, {"champs": all_internvl_champs}, full_ocr_text
    )

    return {
        "n_pages": len(pages),
        "type_document_detecte": doc_type_guess,
        "pays_emetteur_estime": pays_guess,
        "langue_document": langue_guess,
        "qualite_transcription_paddleocr_vl": round(float(np.mean(ocr_quality_scores)), 3) if ocr_quality_scores else 0.0,
        "transcription_paddleocr_vl": full_ocr_text,
        **reconciliation,
    }


## 12. Pipeline complet par client (les deux documents + cross-vérification d'identité)

In [ ]:
def process_client(client_id: str, pdfs: dict) -> dict:
    record = {"client_id": client_id}

    record["justificatif_identite"] = (
        process_pdf_document(pdfs["id_card"]) if pdfs.get("id_card")
        else {"statut": "fichier_absent"}
    )
    record["justificatif_domicile"] = (
        process_pdf_document(pdfs["residence"]) if pdfs.get("residence")
        else {"statut": "fichier_absent"}
    )

    if pdfs.get("id_card") and pdfs.get("residence"):
        record["cross_verification_identite"] = cross_check_identity(
            record["justificatif_identite"].get("champs_qwen", []),
            record["justificatif_domicile"].get("champs_qwen", []),
        )
    else:
        record["cross_verification_identite"] = {"verifiable": False, "detail": "un des deux documents est absent"}

    statuts = [record["justificatif_identite"].get("statut"), record["justificatif_domicile"].get("statut")]
    identite_ok = record["cross_verification_identite"].get("coherent", True)
    if "fichier_absent" in statuts or "a_revoir_manuellement" in statuts or identite_ok is False:
        record["statut_global"] = "a_revoir_manuellement"
    else:
        record["statut_global"] = "certifie"

    return record


## 13. Traitement par lot sur l'ensemble des clients du ZIP

In [ ]:
def batch_process_zip(zip_path: str) -> tuple:
    root = extract_zip(zip_path, EXTRACT_DIR)
    folders = list_client_folders(root)
    print(f"{len(folders)} clients detectes dans l'archive.")

    all_records = []
    for cid, folder in folders.items():
        pdfs = find_target_pdfs(folder)
        try:
            rec = process_client(cid, pdfs)
        except Exception as e:
            rec = {"client_id": cid, "statut_global": "erreur_traitement", "erreur": str(e)}
        all_records.append(rec)
        print(f"client_id={cid:15s} -> statut_global = {rec.get('statut_global')}")

    df = pd.DataFrame(all_records)
    return df, all_records


# Exemple d'execution :
# df_resultats, records_bruts = batch_process_zip(ZIP_PATH)
# df_resultats.to_csv(f"{OUTPUT_DIR}/resultats_recertification_clients.csv", index=False)
# with open(f"{OUTPUT_DIR}/resultats_recertification_clients.json", "w", encoding="utf-8") as f:
#     json.dump(records_bruts, f, ensure_ascii=False, indent=2)
# df_resultats


## Notes de production

- **PaddleOCR-VL** : vérifier dans le `README`/code du modèle livré sur le ModelHub le déclencheur exact de transcription (`"OCR:"` est un exemple générique) et la classe de chargement (`AutoModel` vs une classe dédiée type `PaddleOCRVLForConditionalGeneration`) ; ajuster `extract_raw_text_with_paddleocr_vl` en conséquence.
- **Nommage des fichiers** : `TARGET_FILES` accepte déjà `JUSTIFICATIF IDENTITE`/`JUSTIFICATIF DOMICILE` en plus des anciens alias ; ajouter d'autres variantes si observées en production.
- **Absence de schéma** : la structure `{"champs": [{"libelle_original", "libelle_normalise_fr", "valeur"}]}` reste volontairement libre — aucune liste de champs attendus n'est imposée.
- **Seuils de décision** (`confidence >= 0.5`, `ocr_rate >= 0.7`, `similarite >= 0.75`) sont des points de départ : à recalibrer sur un premier lot de documents annotés manuellement.
- **Traçabilité KYC** : conserver le JSON brut de chaque page (Qwen, InternVL, transcription PaddleOCR-VL) même en cas de certification automatique, pour audit.
- **Performance** : envisager de paralléliser `process_client` sur un pool de workers GPU si le volume de clients est important.
